# Assignment 2 — The Dirty Ledger: audit, cleaning court and before/after evidence

`teaching_prices_messy.csv` is treated as **read-only**. Every cleaning run is made on a copy, and the result is written to a separate `teaching_prices_clean.csv`.

Raw snapshot SHA-256: `5818dad3807487d4e37f764e5d19cbaa152edd78fe4db182eb28b087c8709d35`

**Notebook map**

1. Diagnosis before cleaning — structure, types, missingness, duplicates, key coverage, plausibility.
2. The AI draft, run on a copy — what it changed and what it broke.
3. The court verdicts — keep, modify or reject, with before/after evidence.
4. The final clean — before/after table, row-level change log, verification.
5. What the invalid values would have done to the averages.

The cleaning decisions are also written up in `1_attempt.md`; the full line-by-line trial is in `3_log_reflection.md`.

In [1]:
import hashlib
from itertools import product

import numpy as np
import pandas as pd

RAW_PATH = "teaching_prices_messy.csv"
RAW_SHA256 = hashlib.sha256(open(RAW_PATH, "rb").read()).hexdigest()

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 60)

print("pandas:", pd.__version__)
print("raw file sha256:", RAW_SHA256)

pandas: 3.0.2
raw file sha256: 5818dad3807487d4e37f764e5d19cbaa152edd78fe4db182eb28b087c8709d35


## 1. Diagnosis before changing anything

The file is loaded as text (`dtype=str`, `keep_default_na=False`) so nothing is silently converted before it has been counted. That matters here: if pandas guesses the price column, the comma value and the blank can disappear into a different dtype without the audit seeing them.

In [2]:
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
print("shape (data rows, columns):", raw.shape)
raw.insert(0, "row_id", range(len(raw)))
print("shape with audit row_id:", raw.shape)
print("columns:", raw.columns.tolist())
raw.head(8)

shape (data rows, columns): (97, 6)
shape with audit row_id: (97, 7)
columns: ['row_id', 'date', 'item', 'city', 'price', 'unit', 'data_label']


,row_id,date,item,city,price,unit,data_label
0,0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
5,5,2026-01-01,Pulse C,City 2,,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
6,6,2026-01-01,Sweetener D,City 1,149.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
7,7,2026-01-01,Sweetener D,City 2,155.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [3]:
# Structure: dtype, blanks and distinct values for every column except the audit row id.
structure = pd.DataFrame({
    "dtype": raw.drop(columns="row_id").dtypes.astype(str),
    "non_blank": raw.drop(columns="row_id").apply(lambda s: (s.str.strip() != "").sum()),
    "blank": raw.drop(columns="row_id").apply(lambda s: (s.str.strip() == "").sum()),
    "distinct": raw.drop(columns="row_id").nunique(),
})
structure

,dtype,non_blank,blank,distinct
date,str,97,0,12
item,str,97,0,5
city,str,97,0,2
price,str,96,1,71
unit,str,97,0,1
data_label,str,97,0,1


In [4]:
# Dates: are they text? do they parse? are they evenly spaced?
dates = pd.to_datetime(raw["date"], format="%Y-%m-%d", errors="coerce")
print("unparseable dates:", dates.isna().sum())
print("distinct dates:", dates.nunique())
print("range:", dates.min().date(), "->", dates.max().date())
print("gaps in days:", sorted(dates.drop_duplicates().sort_values().diff().dropna().dt.days.unique()))

unparseable dates: 0
distinct dates: 12
range: 2026-01-01 -> 2026-03-19
gaps in days: [np.int64(7)]


In [5]:
# Labels: the item column is the first surprise.
for col in ["item", "city", "unit", "data_label"]:
    print(f"--- {col}: {raw[col].nunique()} distinct ---")
    print(raw[col].value_counts(dropna=False).to_string())
    print()

--- item: 5 distinct ---
item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
oil b           1

--- city: 2 distinct ---
city
City 1    49
City 2    48

--- unit: 1 distinct ---
unit
PKR per kg (illustrative)    97

--- data_label: 1 distinct ---
data_label
ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS    97



In [6]:
# Price: stored as text. How many cells are not plain numbers?
price_text = raw["price"].str.strip()
strict = pd.to_numeric(price_text, errors="coerce")
parsed_price = pd.to_numeric(price_text.str.replace(",", "", regex=False), errors="coerce")

print("blank price cells:", (price_text == "").sum())
print("non-blank cells that fail a strict parse:", int(((strict.isna()) & (price_text != "")).sum()))
print("raw values with a comma:", price_text[price_text.str.contains(",", regex=False)].tolist())
print()
print(parsed_price.describe().round(2).to_string())

blank price cells: 1
non-blank cells that fail a strict parse: 1
raw values with a comma: ['1,250']

count      96.00
mean      315.46
std      1007.56
min        99.50
25%       142.25
50%       200.00
75%       304.00
max      9999.00


In [7]:
# Duplicates: exact rows and item-city-date keys.
original_cols = [c for c in raw.columns if c != "row_id"]
key = ["date", "item", "city"]

print("exact duplicate rows:", int(raw.duplicated(subset=original_cols).sum()))
print("rows involved in a duplicated key:", int(raw.duplicated(subset=key, keep=False).sum()))
print()
print(raw[raw.duplicated(subset=key, keep=False)].to_string(index=False))

exact duplicate rows: 1
rows involved in a duplicated key: 2

 row_id       date     item   city price                      unit                                               data_label
      8 2026-01-08 Staple A City 1  99.5 PKR per kg (illustrative) ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS
     96 2026-01-08 Staple A City 1  99.5 PKR per kg (illustrative) ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS


In [8]:
# Key coverage against the expected balanced panel: 4 items x 2 cities x 12 dates = 96.
EXPECTED_ITEMS = ["Oil B", "Pulse C", "Staple A", "Sweetener D"]
CITIES = ["City 1", "City 2"]
DATES = pd.date_range("2026-01-01", "2026-03-19", freq="7D")
expected_keys = set(product(DATES, EXPECTED_ITEMS, CITIES))

raw_parsed = raw.copy()
raw_parsed["date"] = pd.to_datetime(raw_parsed["date"], format="%Y-%m-%d")
raw_parsed["price"] = pd.to_numeric(
    raw_parsed["price"].str.replace(",", "", regex=False), errors="coerce"
)

actual_keys = set(raw_parsed[["date", "item", "city"]].itertuples(index=False, name=None))

print("expected keys:", len(expected_keys))
print("unique keys in file:", len(actual_keys))
print("canonical keys present:", len(actual_keys & expected_keys), "/", len(expected_keys))
print("missing keys:", sorted(expected_keys - actual_keys))
print("unexpected keys:", sorted(actual_keys - expected_keys))

expected keys: 96
unique keys in file: 96
canonical keys present: 95 / 96
missing keys: [(Timestamp('2026-02-05 00:00:00'), 'Staple A', 'City 1')]
unexpected keys: [(Timestamp('2026-02-05 00:00:00'), 'oil b', 'City 1')]


In [9]:
# The 6.0 city gap is a built-in consistency check. One table reveals two bad values.
pair = raw_parsed.pivot_table(index=["item", "date"], columns="city", values="price")
gaps = (pair["City 2"] - pair["City 1"]).dropna()

print("City 2 - City 1, value counts:")
print(gaps.value_counts().to_string())
print()
print("The two exceptions:")
print(gaps[gaps != 6.0].to_string())

City 2 - City 1, value counts:
 6.0       44
 1147.0     1
-9835.5     1

The two exceptions:
item         date      
Staple A     2026-01-15    1147.0
Sweetener D  2026-01-22   -9835.5


In [10]:
# Neighbourhood evidence for the three suspicious values.
def series_around(item, city, centre, weeks=2):
    s = raw_parsed[(raw_parsed["item"] == item) & (raw_parsed["city"] == city)].sort_values("date")
    lo = pd.Timestamp(centre) - pd.Timedelta(weeks=weeks)
    hi = pd.Timestamp(centre) + pd.Timedelta(weeks=weeks)
    return s[s["date"].between(lo, hi)][["row_id", "date", "item", "city", "price"]]

print("A. Staple A / City 2 around 2026-01-15 (the 1,250)")
print(series_around("Staple A", "City 2", "2026-01-15").to_string(index=False))
print()
print("B. Sweetener D / City 1 around 2026-01-22 (the 9,999)")
print(series_around("Sweetener D", "City 1", "2026-01-22").to_string(index=False))
print()
print("C. All labels at 2026-02-05 / City 1 (the 'oil b' row)")
print(
    raw_parsed[(raw_parsed["date"] == "2026-02-05") & (raw_parsed["city"] == "City 1")]
    [["row_id", "item", "price"]]
    .sort_values("item")
    .to_string(index=False)
)
print()
print("D. Staple A / City 1 around 2026-02-05 (its missing cell)")
print(series_around("Staple A", "City 1", "2026-02-05").to_string(index=False))
print()
print("E. Oil B / City 1 around 2026-02-05 (where 108.5 does not fit)")
print(series_around("Oil B", "City 1", "2026-02-05").to_string(index=False))

A. Staple A / City 2 around 2026-01-15 (the 1,250)
 row_id       date     item   city  price
      1 2026-01-01 Staple A City 2  107.0
      9 2026-01-08 Staple A City 2  105.5
     17 2026-01-15 Staple A City 2 1250.0
     25 2026-01-22 Staple A City 2  112.5
     33 2026-01-29 Staple A City 2  111.0

B. Sweetener D / City 1 around 2026-01-22 (the 9,999)
 row_id       date        item   city  price
     14 2026-01-08 Sweetener D City 1  153.5
     22 2026-01-15 Sweetener D City 1  153.0
     30 2026-01-22 Sweetener D City 1 9999.0
     38 2026-01-29 Sweetener D City 1  162.0
     46 2026-02-05 Sweetener D City 1  161.5

C. All labels at 2026-02-05 / City 1 (the 'oil b' row)
 row_id        item  price
     42       Oil B  318.0
     44     Pulse C  202.5
     46 Sweetener D  161.5
     40       oil b  108.5

D. Staple A / City 1 around 2026-02-05 (its missing cell)
 row_id       date     item   city  price
     24 2026-01-22 Staple A City 1  106.5
     32 2026-01-29 Staple A City 1  10

In [11]:
# The clincher: Staple A / City 2 on 2026-02-05 minus the usual 6.0 gap.
staple_c2 = raw_parsed[
    (raw_parsed["item"] == "Staple A")
    & (raw_parsed["city"] == "City 2")
    & (raw_parsed["date"] == "2026-02-05")
]["price"].item()

print("Staple A / City 2 / 2026-02-05:", staple_c2)
print("minus the usual 6.0 gap:", staple_c2 - 6.0)
print("the 'oil b' row at City 1 / 2026-02-05:", raw_parsed.loc[raw_parsed["row_id"] == 40, "price"].item())

Staple A / City 2 / 2026-02-05: 114.5
minus the usual 6.0 gap: 108.5
the 'oil b' row at City 1 / 2026-02-05: 108.5


### Diagnosis table

| # | Problem | Code that revealed it | Planned decision | Reason |
|---|---|---|---|---|
| **D1** | `price` is text, not numeric; one blank and one comma-formatted value are not plain numbers | `raw.dtypes`; a strict `pd.to_numeric(..., errors="coerce")`; `(price_text == "").sum()` | **correct**: parse to float, strip the thousands comma, blanks → `NaN` | `describe()` and means are meaningless on text; coercing without counting would hide two bad cells |
| **D2** | `1,250` at Staple A / City 2 / 2026-01-15 is ~11× the local level | `pivot_table` city-gap check; `series_around` | **flag**: price → `NaN`, keep row | City 2 − City 1 is 6.0 in 44 of 46 complete pairs; this pair is 1147.0. Neighbours are 105.5 and 112.5; the gap implies 109.0, but that is an inference from the paired city, not a value observed in this row. Flag, do not overwrite |
| **D3** | `9999` at Sweetener D / City 1 / 2026-01-22 is a sentinel | same gap check; `describe()` max | **flag**: price → `NaN`, keep row | neighbours 153.0 and 162.0; City 2 is 163.5; the gap implies 157.5, but flag rather than overwrite. 9999 is ~63× the level and would inflate the mean from ~167 to ~577 |
| **D4** | blank price at Pulse C / City 2 / 2026-01-01 | `(price_text == "").sum()` | **retain**: keep row, leave `NaN` | dropping the row loses a valid observation and breaks the 12-date panel; the gap implies 206.0, but I leave it missing rather than impute and shrink the variance |
| **D5** | `oil b` at City 1 / 2026-02-05 is a mislabeled **Staple A** observation | `value_counts()`; item ranges; key grid; city gap | **investigate → correct** to `Staple A` | 108.5 is less than half of Oil B's lowest value (298); it exactly fills the only missing canonical key; Staple A / City 1 runs 105.0 → 108.5 → 107.0; 114.5 − 6.0 = 108.5. `.str.title()` to Oil B would be wrong |
| **D6** | exact duplicate row at the end | `raw.duplicated()` | **exclude**: drop one copy | `row_id` 8 and `row_id` 96 are identical on all six columns; keeping it double-counts 99.5 and masks the missing 2026-02-05 Staple A cell |
| **D7** | 5 item labels for 4 items; one missing key, one extra key | key grid vs 4 × 2 × 12 | **correct**: canonical four labels | the extra label is the D5 row; after the fix the panel is balanced |
| **D8** | `date` is text | `raw.dtypes`; `to_datetime` | **correct**: parse ISO dates | 0 unparseable, 12 distinct dates, exactly 7 days apart; strict parse is safe |
| **D9** | `unit` and `data_label` are constant | `nunique()` | **retain** both | the unit interprets `price`; the label is the teaching-data disclaimer, not redundant data |
| **D10** | row order is not panel order; duplicate appended at the end | `head()` / `tail()` | **retain** values, sort by date/item/city | cosmetic, but makes the panel and the duplicate visible |

**Checks that passed:** 97 data rows, 6 columns; 2 cities only; 12 ISO dates exactly 7 days apart; no duplicate dates; no negative or zero prices; no other missing values.

## 2. The ASK and the AI draft

**What I described to the AI (in words, no rows pasted):** one table, 97 data rows, six columns (`date`, `item`, `city`, `price`, `unit`, `data_label`); four items in two cities over twelve weekly dates (2026-01-01 → 2026-03-19); `price` is text with one blank, one quoted `"1,250"` and one `9999`; one item label is `oil b`; one exact duplicate row; and one missing item-city-date key. I asked for pandas cleaning code.

The draft below is the code the AI returned. It is run on a **copy** — never on `teaching_prices_messy.csv` — and each line is put on trial afterwards.

```python
def ai_clean_v1(path):
    df = pd.read_csv(path)                                              # L1
    df.columns = [c.strip().lower() for c in df.columns]                # L2
    df = df.dropna()                                                    # L3
    df["item"] = df["item"].str.strip().str.title()                     # L4
    df["price"] = df["price"].astype(str).str.replace(",", "", regex=False)  # L5
    df["price"] = pd.to_numeric(df["price"], errors="coerce")           # L6
    df = df.drop_duplicates()                                           # L7
    q1, q3 = df["price"].quantile([0.25, 0.75])                         # L8
    iqr = q3 - q1                                                       # L9
    df = df[df["price"].between(q1 - 3 * iqr, q3 + 3 * iqr)]            # L10
    df["date"] = pd.to_datetime(df["date"])                             # L11
    df["price"] = df.groupby(["item", "city"])["price"].transform(      # L12
        lambda s: s.fillna(s.mean())                                    # L13
    )
    return df.sort_values(["date", "item", "city"]).reset_index(drop=True)  # L14
```

In [12]:
def ai_clean_v1(path):
    df = pd.read_csv(path)                                              # L1
    df.columns = [c.strip().lower() for c in df.columns]                # L2
    df = df.dropna()                                                    # L3
    df["item"] = df["item"].str.strip().str.title()                     # L4
    df["price"] = df["price"].astype(str).str.replace(",", "", regex=False)  # L5
    df["price"] = pd.to_numeric(df["price"], errors="coerce")           # L6
    df = df.drop_duplicates()                                           # L7
    q1, q3 = df["price"].quantile([0.25, 0.75])                         # L8
    iqr = q3 - q1                                                       # L9
    df = df[df["price"].between(q1 - 3 * iqr, q3 + 3 * iqr)]            # L10
    df["date"] = pd.to_datetime(df["date"])                             # L11
    df["price"] = df.groupby(["item", "city"])["price"].transform(      # L12
        lambda s: s.fillna(s.mean())                                    # L13
    )
    return df.sort_values(["date", "item", "city"]).reset_index(drop=True)  # L14

v1 = ai_clean_v1(RAW_PATH)
print("shape:", v1.shape)
print("dtypes:")
print(v1.dtypes.to_string())
print()
print("missing values:")
print(v1.isna().sum().to_string())

shape: (93, 6)
dtypes:
date          datetime64[us]
item                     str
city                     str
price                float64
unit                     str
data_label               str

missing values:
date          0
item          0
city          0
price         0
unit          0
data_label    0


In [13]:
# Before/after profile. The raw row 5 (blank) is counted separately below.
def panel_profile(df):
    p = df["price"]
    return pd.Series({
        "rows": len(df),
        "item labels": df["item"].nunique(),
        "cities": df["city"].nunique(),
        "dates": df["date"].nunique(),
        "unique keys": df[["date", "item", "city"]].drop_duplicates().shape[0],
        "duplicate key rows": int(df.duplicated(["date", "item", "city"]).sum()),
        "missing prices": int(p.isna().sum()),
        "min price": p.min(),
        "max price": p.max(),
    })

profile = pd.DataFrame({
    "raw": panel_profile(raw_parsed),
    "AI v1": panel_profile(v1),
})

def format_profile(df):
    out = df.copy()
    for col in out.columns:
        out[col] = out[col].map(
            lambda x: f"{x:,.0f}" if float(x).is_integer() else f"{x:,.2f}"
        )
    return out

format_profile(profile)

,raw,AI v1
rows,97,93
item labels,5,4
cities,2,2
dates,12,12
unique keys,96,92
duplicate key rows,1,1
missing prices,1,0
min price,99.50,99.50
max price,"9,999",350


In [14]:
# Item counts: the balanced panel is the first thing the AI draft breaks.
counts = pd.DataFrame({
    "raw": raw_parsed["item"].value_counts(),
    "AI v1": v1["item"].value_counts(),
    "my clean (expected)": pd.Series(24, index=EXPECTED_ITEMS),
}).fillna(0).astype(int)
counts = counts.reindex(EXPECTED_ITEMS + ["oil b"]).fillna(0).astype(int)
counts

,raw,AI v1,my clean (expected)
Oil B,24,25,24
Pulse C,24,23,24
Staple A,24,22,24
Sweetener D,24,23,24
oil b,1,0,0


In [15]:
# Which real observations did v1 lose, and which duplicate key did it create?
checks = pd.DataFrame([
    ("2026-01-01", "Pulse C", "City 2"),
    ("2026-01-15", "Staple A", "City 2"),
    ("2026-01-22", "Sweetener D", "City 1"),
    ("2026-02-05", "Staple A", "City 1"),
    ("2026-02-05", "Oil B", "City 1"),
], columns=["date", "item", "city"])
checks["date"] = pd.to_datetime(checks["date"])

checks["v1_rows"] = checks.apply(
    lambda r: len(v1[(v1["date"] == r["date"]) & (v1["item"] == r["item"]) & (v1["city"] == r["city"])]),
    axis=1,
)
checks["v1_prices"] = checks.apply(
    lambda r: v1[(v1["date"] == r["date"]) & (v1["item"] == r["item"]) & (v1["city"] == r["city"])]["price"].tolist(),
    axis=1,
)
checks

,date,item,city,v1_rows,v1_prices
0,2026-01-01,Pulse C,City 2,0,[]
1,2026-01-15,Staple A,City 2,0,[]
2,2026-01-22,Sweetener D,City 1,0,[]
3,2026-02-05,Staple A,City 1,0,[]
4,2026-02-05,Oil B,City 1,2,"[108.5, 318.0]"


In [16]:
print("v1 duplicate-key rows:")
print(v1[v1.duplicated(["date", "item", "city"], keep=False)].to_string(index=False))
print()
print("v1 mean by item (compare with the clean table in section 4):")
print(v1.groupby("item")["price"].agg(["count", "mean", "min", "max"]).round(2).to_string())

v1 duplicate-key rows:
      date  item   city  price                      unit                                               data_label
2026-02-05 Oil B City 1  108.5 PKR per kg (illustrative) ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS
2026-02-05 Oil B City 1  318.0 PKR per kg (illustrative) ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS

v1 mean by item (compare with the clean table in section 4):
             count    mean    min    max
item                                    
Oil B           25  316.18  108.5  350.0
Pulse C         23  205.91  200.0  213.5
Staple A        22  111.39   99.5  122.0
Sweetener D     23  167.15  149.0  184.5


### What the AI draft broke

| Check | Raw | AI v1 | Why |
|---|---:|---:|---|
| Rows | 97 | **93** | `dropna()` deletes the blank Pulse C row; the pooled IQR filter deletes the Staple A and Sweetener D rows; one exact duplicate is dropped |
| Item counts | 24 / 24 / 24 / 24 + 1 `oil b` | **25 / 23 / 22 / 23** | `oil b` is title-cased into Oil B, so Oil B absorbs the Staple A observation; Pulse C and Sweetener D lose the deleted rows |
| Unique keys | 96 (95 canonical + 1 spurious) | **92** | three real keys deleted; one duplicate key created at (2026-02-05, Oil B, City 1) |
| Duplicate key rows | 1 | **1** | `drop_duplicates()` ran after the wrong label, so it cannot remove the duplicate key |
| Missing prices | 1 blank | **0** | not fixed: the rows that held the missing/invalid prices were deleted |
| Max price | 9999 | 350 | the two invalid values were removed by deletion, not by flagging |
| Staple A mean | 158.33 (1250 included) | 111.39 (22 rows) | the mean is close only because the row was deleted; the observation is gone |

The important point is not that v1 is short or obviously wrong. It is that the table *looks* clean — 0 missing, max 350 — while three valid observations have disappeared and Oil B has been contaminated with a 108.5 price. Counts in the table are in the order Oil B / Pulse C / Staple A / Sweetener D.

## 3. The court verdicts (summary)

The full trial is in `3_log_reflection.md`. The short version:

| Line | Code | Verdict | Reason |
|---|---|---|---|
| L1 | `read_csv` | **keep** | loads 97 × 6; no data change |
| L2 | lowercase/strip headers | **keep** | columns are already clean; idempotent |
| L3 | `dropna()` | **reject** | removes a valid observation (Pulse C / City 2 / 2026-01-01) |
| L4 | `.str.title()` | **reject** | turns the `oil b` mislabel into a wrong Oil B price; creates a duplicate key |
| L5 | strip commas | **keep** | correct parse of `"1,250"`; plausibility is a separate check |
| L6 | `to_numeric(errors="coerce")` | **keep** | blank → `NaN`, text → number, no exception; count the nulls |
| L7 | `drop_duplicates()` | **modify** | correct only after the label fix, and must assert key uniqueness |
| L8–L9 | pooled quantiles / IQR | **keep** (arithmetic) | correct arithmetic, wrong population (items at different levels) |
| L10 | pooled 3×IQR filter | **reject** | fences (−308, 763) delete whole rows and still leave the 108.5 Oil B value |
| L11 | `to_datetime` | **keep** | 12 dates, 0 NaT, 7-day gaps |
| L12–L13 | group-mean fill | **reject** | invents prices and hides missingness; it has nothing to fill only because L3/L10 deleted the rows |
| L14 | sort | **keep** | panel order; no value change |

## 4. The final clean

The final function keeps the observation, fixes only what the evidence supports, and records every change. The rules are:

- parse, don't guess: `price_raw` is kept beside the parsed `price`;
- one documented label fix: `oil b` → `Staple A`;
- flag invalid prices with a **per-item** 3×IQR fence, set them to `NaN`, keep the row;
- drop only exact duplicate rows;
- never impute a price;
- sort into panel order.

In [17]:
def clean_ledger(path):
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    raw.insert(0, "row_id", range(len(raw)))
    df = raw.copy()

    # 1. Trim text columns.
    for c in ["date", "item", "city", "unit", "data_label"]:
        df[c] = df[c].str.strip()

    # 2. Parse dates strictly (ISO format is unambiguous here).
    df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise")

    # 3. Keep the raw price text, then parse: comma is a thousands separator.
    #    Blank / non-numeric becomes NaN, never zero.
    df["price_raw"] = df["price"].str.strip()
    df["price"] = pd.to_numeric(
        df["price_raw"].str.replace(",", "", regex=False), errors="coerce"
    )

    # 4. The one label fix, with the evidence in 1_attempt.md D5:
    #    108.5 is Staple A's missing 2026-02-05 / City 1 value, not an Oil B price.
    item_map = {
        "Staple A": "Staple A",
        "Oil B": "Oil B",
        "Pulse C": "Pulse C",
        "Sweetener D": "Sweetener D",
        "oil b": "Staple A",
    }
    df["item"] = df["item"].map(item_map)
    unmapped = df["item"].isna()
    if unmapped.any():
        raise ValueError("Unmapped item label(s): " + str(sorted(raw.loc[unmapped, "item"].unique())))

    # 5. Drop only exact duplicates, keeping the first copy.
    before = len(df)
    df = df.drop_duplicates(
        subset=["date", "item", "city", "price_raw", "unit", "data_label"],
        keep="first",
    ).copy()
    duplicates_removed = before - len(df)

    # 6. Per-item 3xIQR plausibility fence. This is not a distribution test:
    #    the four items sit at different price levels, so a pooled fence is wrong.
    df["price_flag"] = False
    df["flag_reason"] = ""
    for item, g in df.groupby("item"):
        q1, q3 = g["price"].quantile([0.25, 0.75])
        iqr = q3 - q1
        low, high = q1 - 3 * iqr, q3 + 3 * iqr
        bad = g.index[(g["price"] < low) | (g["price"] > high)]
        df.loc[bad, "price_flag"] = True
        df.loc[bad, "flag_reason"] = f"{item}: outside 3xIQR [{low:.1f}, {high:.1f}]"

    # 7. Flag means "do not use this number", not "delete this observation".
    df.loc[df["price_flag"], "price"] = np.nan

    # 8. Sort into panel order.
    df = df.sort_values(["date", "item", "city"]).reset_index(drop=True)
    return df, duplicates_removed

clean, duplicates_removed = clean_ledger(RAW_PATH)
print("rows:", len(clean))
print("exact duplicates removed:", duplicates_removed)
print()
print(clean[["row_id", "date", "item", "city", "price", "price_flag", "flag_reason"]].head(10).to_string(index=False))

rows: 96
exact duplicates removed: 1

 row_id       date        item   city  price  price_flag flag_reason
      2 2026-01-01       Oil B City 1  298.0       False            
      3 2026-01-01       Oil B City 2  304.0       False            
      4 2026-01-01     Pulse C City 1  200.0       False            
      5 2026-01-01     Pulse C City 2    NaN       False            
      0 2026-01-01    Staple A City 1  101.0       False            
      1 2026-01-01    Staple A City 2  107.0       False            
      6 2026-01-01 Sweetener D City 1  149.0       False            
      7 2026-01-01 Sweetener D City 2  155.0       False            
     10 2026-01-08       Oil B City 1  304.0       False            
     11 2026-01-08       Oil B City 2  310.0       False            


In [18]:
# Every missing price, with the raw text kept beside it.
missing = clean[clean["price"].isna()].copy()
missing["date"] = missing["date"].dt.strftime("%Y-%m-%d")
missing["reason"] = np.where(
    missing["price_flag"], missing["flag_reason"], "blank in source; retained"
)
missing[["row_id", "date", "item", "city", "price_raw", "price", "price_flag", "reason"]]

,row_id,date,item,city,price_raw,price,price_flag,reason
3,5,2026-01-01,Pulse C,City 2,,NaN,False,blank in source; retained
21,17,2026-01-15,Staple A,City 2,"1,250",NaN,True,"Staple A: outside 3xIQR [79.1, 143.9]"
30,30,2026-01-22,Sweetener D,City 1,9999,NaN,True,"Sweetener D: outside 3xIQR [119.4, 216.5]"


In [19]:
# Before/after table.
before_after = pd.DataFrame({
    "raw": panel_profile(raw_parsed),
    "clean": panel_profile(clean),
})
format_profile(before_after)

,raw,clean
rows,97,96
item labels,5,4
cities,2,2
dates,12,12
unique keys,96,96
duplicate key rows,1,0
missing prices,1,3
min price,99.50,99.50
max price,"9,999",350


In [20]:
# describe() before and after, on the same price measure.
pd.DataFrame({
    "raw": raw_parsed["price"].describe(),
    "AI v1": v1["price"].describe(),
    "clean": clean["price"].describe(),
}).round(2)

,raw,AI v1,clean
count,96.00,93.00,93.00
mean,315.46,203.61,203.61
std,1007.56,79.92,79.92
min,99.50,99.50,99.50
25%,142.25,149.00,149.00
50%,200.00,200.00,200.00
75%,304.00,298.00,298.00
max,9999.00,350.00,350.00


`describe()` alone would not catch the v1 label error: v1 and clean both have the same 93 numeric values, so their pooled count, mean and standard deviation are identical. The damage is in **which item owns 108.5**, which only the item-level counts and means reveal. That is why the court uses both the pooled describe and the per-item table.

In [21]:
# Every item must appear 4 x 3 = 12 times per city and 24 times overall.
expected_counts = pd.Series(24, index=EXPECTED_ITEMS)
actual_counts = clean.groupby("item").size()

pd.DataFrame({
    "expected": expected_counts,
    "actual": actual_counts,
    "ok": expected_counts == actual_counts,
})

,expected,actual,ok
Oil B,24,24,True
Pulse C,24,24,True
Staple A,24,24,True
Sweetener D,24,24,True


In [22]:
# Row-level change log: exactly what changed, and nothing else.
raw_small = raw[["row_id", "date", "item", "price"]].rename(
    columns={"item": "item_raw", "price": "price_text"}
)
clean_small = clean[["row_id", "item", "price", "price_flag", "flag_reason"]].rename(
    columns={"item": "item_clean", "price": "price_clean"}
)
changes = raw_small.merge(clean_small, on="row_id", how="left", indicator=True)
changes["price_before_num"] = pd.to_numeric(
    changes["price_text"].str.replace(",", "", regex=False), errors="coerce"
)

def classify(row):
    if row["_merge"] == "left_only":
        return "dropped: exact duplicate"
    if row["item_raw"] != row["item_clean"]:
        return "label corrected: oil b -> Staple A"
    if pd.isna(row["price_before_num"]) and pd.isna(row["price_clean"]):
        return "retained: missing price"
    if pd.notna(row["price_before_num"]) and pd.isna(row["price_clean"]):
        return "flagged: invalid price -> missing"
    return "unchanged"

changes["action"] = changes.apply(classify, axis=1)
changed_rows = changes[changes["action"] != "unchanged"].copy()

changed_rows[[
    "row_id", "date", "item_raw", "item_clean", "price_text",
    "price_before_num", "price_clean", "price_flag", "flag_reason", "action"
]]

,row_id,date,item_raw,item_clean,price_text,price_before_num,price_clean,price_flag,flag_reason,action
5,5,2026-01-01,Pulse C,Pulse C,,NaN,NaN,False,,retained: missing price
17,17,2026-01-15,Staple A,Staple A,"1,250",1250.0,NaN,True,"Staple A: outside 3xIQR [79.1, 143.9]",flagged: invalid price -> missing
30,30,2026-01-22,Sweetener D,Sweetener D,9999,9999.0,NaN,True,"Sweetener D: outside 3xIQR [119.4, 216.5]",flagged: invalid price -> missing
40,40,2026-02-05,oil b,Staple A,108.5,108.5,108.5,False,,label corrected: oil b -> Staple A
96,96,2026-01-08,Staple A,NaN,99.5,99.5,NaN,NaN,NaN,dropped: exact duplicate


In [23]:
# Assert that only the five documented rows changed, and no other cell moved.
documented = {5, 17, 30, 40, 96}
assert set(changed_rows["row_id"]) == documented, set(changed_rows["row_id"])

untouched = changes[(changes["_merge"] == "both") & ~changes["row_id"].isin(documented)]
assert untouched["item_raw"].eq(untouched["item_clean"]).all()
assert (
    untouched["price_before_num"].fillna(-999)
    == untouched["price_clean"].fillna(-999)
).all()

print("PASS: only rows", sorted(documented), "changed; every other value is identical.")

PASS: only rows [5, 17, 30, 40, 96] changed; every other value is identical.


In [24]:
# Verification: balanced panel, complete keys, no duplicate keys, plausible prices.
expected_dates = set(DATES)
clean_keys = set(clean[["date", "item", "city"]].itertuples(index=False, name=None))

checks = {
    "96 rows (4 x 2 x 12)": len(clean) == 96,
    "4 canonical item labels": set(clean["item"]) == set(EXPECTED_ITEMS),
    "12 weekly dates": set(clean["date"]) == expected_dates,
    "no duplicate keys": int(clean.duplicated(["date", "item", "city"]).sum()) == 0,
    "96 / 96 canonical keys": clean_keys == expected_keys,
    "each item 24 rows": bool(clean.groupby("item").size().eq(24).all()),
    "each item-city 12 rows": bool(clean.groupby(["item", "city"]).size().eq(12).all()),
    "each date 8 rows": bool(clean.groupby("date").size().eq(8).all()),
    "only 3 missing prices": int(clean["price"].isna().sum()) == 3,
    "prices in 99.5-350": bool(clean["price"].dropna().between(99.5, 350).all()),
}

for name, ok in checks.items():
    print(("PASS" if ok else "FAIL"), "-", name)

assert all(checks.values())

wide = clean.pivot_table(index=["item", "date"], columns="city", values="price")
gap = (wide["City 2"] - wide["City 1"]).dropna()
print()
print("complete city pairs:", len(gap))
print("distinct City 2 - City 1 gaps:", gap.round(6).unique())
assert gap.round(6).nunique() == 1 and gap.round(6).iloc[0] == 6.0
print("PASS: the 6.0 city gap holds for every complete pair.")

PASS - 96 rows (4 x 2 x 12)
PASS - 4 canonical item labels
PASS - 12 weekly dates
PASS - no duplicate keys
PASS - 96 / 96 canonical keys
PASS - each item 24 rows
PASS - each item-city 12 rows
PASS - each date 8 rows
PASS - only 3 missing prices
PASS - prices in 99.5-350

complete city pairs: 45
distinct City 2 - City 1 gaps: [6.]
PASS: the 6.0 city gap holds for every complete pair.


### Before/after, read as a table

| Check | Raw | Clean | What happened |
|---|---:|---:|---|
| Rows | 97 | **96** | 1 exact duplicate removed; all 96 real observations kept |
| Item labels | 5 | **4** | `oil b` → `Staple A` with evidence (D5) |
| Item counts | 24 / 24 / 24 / 24 + 1 `oil b` | **24 / 24 / 24 / 24** | balanced panel |
| Unique keys | 96 (95 canonical + 1 spurious) | **96 canonical** | missing key recovered, spurious key removed |
| Duplicate key rows | 1 | **0** | exact duplicate dropped after the label fix |
| Missing prices | 1 blank | **3** (1 blank + 2 flagged) | flags are visible, not hidden by deletion or imputation |
| Max price | 9999 | **350** | 1,250 and 9,999 are `NaN`, not prices |
| City 2 − City 1 | 44 × 6.0 + 2 impossible gaps | **6.0 for all 45 complete pairs** | the two impossible gaps are the two flagged values |
| Staple A mean | 158.33 | **111.26** | 1,250 no longer pulls the mean up |
| Sweetener D mean | 576.81 | **167.15** | 9,999 no longer pulls the mean up |

Counts in the table are in the order Oil B / Pulse C / Staple A / Sweetener D. Raw label counts are 24 / 24 / 24 / 24 plus 1 `oil b`. Recoding `oil b` to Staple A and dropping the exact duplicate gives 24 Staple A rows; the other three items stay at 24. The clean file has 24 rows per item and 12 per item-city.

## 5. What the invalid values would have done

The two flagged values are not just "outliers" to trim. They are large enough to change the story of the item. This table holds the 96-row panel fixed and only puts the invalid values back into their rows; it shows why flagging rather than deleting or ignoring matters. (The raw means in the previous table use the raw rows with `oil b` still separate, so they differ slightly from the fixed-panel means here.)

In [25]:
with_bad = clean.copy()
with_bad["price_with_bad"] = with_bad["price"]
bad_mask = with_bad["price_flag"]
with_bad.loc[bad_mask, "price_with_bad"] = pd.to_numeric(
    with_bad.loc[bad_mask, "price_raw"].str.replace(",", "", regex=False)
)

sensitivity = pd.DataFrame({
    "n (rows kept)": clean.groupby("item").size(),
    "mean with invalid values": with_bad.groupby("item")["price_with_bad"].mean(),
    "mean after flagging": clean.groupby("item")["price"].mean(),
}).round(2)
sensitivity["distortion"] = (
    sensitivity["mean with invalid values"] - sensitivity["mean after flagging"]
).round(2)
sensitivity

,n (rows kept),mean with invalid values,mean after flagging,distortion
item,,,,
Oil B,24,324.83,324.83,0.00
Pulse C,24,205.91,205.91,0.00
Staple A,24,158.71,111.26,47.45
Sweetener D,24,576.81,167.15,409.66


## 6. Export and data dictionary

The exported clean file keeps the original six columns. The audit columns (`row_id`, `price_raw`, `price_flag`, `flag_reason`) stay in the notebook; the missing rows and their reasons are also in `cleaning_flags.csv`. They are not in the analysis-ready file.

In [26]:
clean_out = clean[["date", "item", "city", "price", "unit", "data_label"]].copy()
clean_out["date"] = clean_out["date"].dt.strftime("%Y-%m-%d")
clean_out.to_csv("teaching_prices_clean.csv", index=False)

flags_out = clean.loc[clean["price"].isna(), [
    "row_id", "date", "item", "city", "price_raw", "price", "price_flag", "flag_reason"
]].copy()
flags_out["date"] = flags_out["date"].dt.strftime("%Y-%m-%d")
flags_out["reason"] = np.where(
    flags_out["price_flag"], flags_out["flag_reason"], "blank in source; retained"
)
flags_out = flags_out.drop(columns="flag_reason")
flags_out.to_csv("cleaning_flags.csv", index=False)

print("wrote teaching_prices_clean.csv:", len(clean_out), "rows")
print("wrote cleaning_flags.csv:", len(flags_out), "rows")

wrote teaching_prices_clean.csv: 96 rows
wrote cleaning_flags.csv: 3 rows


In [27]:
pd.DataFrame([
    ("date", "date", "Weekly observation date, every Thursday, 2026-01-01 to 2026-03-19"),
    ("item", "category", "4 items: Oil B, Pulse C, Staple A, Sweetener D"),
    ("city", "category", "City 1, City 2"),
    ("price", "float", "PKR per kg (illustrative); 3 missing after the audit"),
    ("unit", "text (constant)", "PKR per kg (illustrative)"),
    ("data_label", "text (constant)", "ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS"),
], columns=["column", "type", "meaning"])

,column,type,meaning
0,date,date,"Weekly observation date, every Thursday, 2026-..."
1,item,category,"4 items: Oil B, Pulse C, Staple A, Sweetener D"
2,city,category,"City 1, City 2"
3,price,float,PKR per kg (illustrative); 3 missing after the...
4,unit,text (constant),PKR per kg (illustrative)
5,data_label,text (constant),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


## 7. Summary

- **Raw:** 97 rows, 6 columns, 5 item labels, 1 blank price, 1 comma-formatted value, 1 sentinel, 1 exact duplicate, 1 mislabeled row, 1 missing key.
- **Clean:** 96 rows (4 items × 2 cities × 12 dates), 24 rows per item, 12 per item-city, 8 per date, 96/96 canonical keys, 0 duplicate keys, 3 missing prices.
- **Changes:** 1 exact duplicate dropped; 1 label corrected (`oil b` → `Staple A`); 2 invalid prices flagged to `NaN`; 1 blank price retained as `NaN`; no imputation; no other cell changed (asserted above).
- **Consistency:** City 2 − City 1 = 6.0 for all 45 complete pairs; every item series sits inside 99.5–350.
- **AI draft:** looked clean (93 rows, 0 missing, max 350) but deleted three valid observations, gave Oil B a 108.5 price, left a duplicate key and left the panel unbalanced. The court kept its parse lines and rejected its deletion, title-casing, pooled filtering and imputation.